In [1]:
import cv2
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from torchvision import transforms

In [2]:
ECHONET_DIR = Path("../../data/raw/EchoNet-Dynamic")
filelist = pd.read_csv(ECHONET_DIR/"FileList.csv")
tracings = pd.read_csv(ECHONET_DIR/"VolumeTracings.csv")
filelist.head(), tracings.head()

(             FileName         EF         ESV         EDV  FrameHeight  \
 0  0X100009310A3BD7FC  78.498406   14.881368   69.210534          112   
 1  0X1002E8FBACD08477  59.101988   40.383876   98.742884          112   
 2  0X1005D03EED19C65B  62.363798   14.267784   37.909734          112   
 3  0X10075961BC11C88E  54.545097   33.143084   72.914210          112   
 4  0X10094BA0A028EAC3  24.887742  127.581945  169.855024          112   
 
    FrameWidth  FPS  NumberOfFrames  Split  
 0         112   50             174    VAL  
 1         112   50             215  TRAIN  
 2         112   50             104  TRAIN  
 3         112   55             122  TRAIN  
 4         112   52             207    VAL  ,
                  FileName         X1         Y1         X2         Y2  Frame
 0  0X100009310A3BD7FC.avi  51.260417  15.348958  64.932292  69.125000     46
 1  0X100009310A3BD7FC.avi  50.037611  17.167841  53.367222  16.321330     46
 2  0X100009310A3BD7FC.avi  49.157378  20.407629 

In [16]:
echo_tfm = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

In [17]:
def ef_to_severity(ef):
    if ef >= 50:
        return 0   # Normal
    elif ef >= 40:
        return 1   # Mild
    else:
        return 2   # Severe

In [18]:
def get_annotated_frames(video_name,tracings):
    video_tracings = tracings[tracings["FileName"] ==video_name]
    frames = sorted(video_tracings["Frame"].unique())

    if len(frames)>2:
        raise ValueError(f"{video_name} has {len(frames)} annotated frames: {frames}")

    return frames

In [19]:
def read_two_frames(video_path, frame_indices):
    cap = cv2.VideoCapture(str(video_path))

    frames = []

    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()

        if not ok:
            cap.release()
            raise RuntimeError(f"Could not read frame {idx} from {video_path}")

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(echo_tfm(frame_rgb))

    cap.release()

    return torch.stack(frames)

In [22]:
row = filelist.iloc[0]

video_name = row["FileName"] +".avi"
video_path = ECHONET_DIR / "Videos" / video_name

frames = get_annotated_frames(video_name, tracings)
print("Annotated frames:", frames)

tensor = read_two_frames(video_path, frames)

print(tensor.shape)
print("EF:", row["EF"])
print("Severity:", ef_to_severity(row["EF"]))

Annotated frames: [np.int64(46), np.int64(61)]
torch.Size([2, 3, 224, 224])
EF: 78.49840597
Severity: 0


In [21]:
# print("video_name from filelist:")
# print(repr(video_name))

# print("\nFirst 5 tracing filenames:")
# print(tracings["FileName"].head().apply(repr).to_list())

# print("\nDoes exact match exist?")
# print((tracings["FileName"] == video_name).sum())

video_name from filelist:
'0X100009310A3BD7FC'

First 5 tracing filenames:
["'0X100009310A3BD7FC.avi'", "'0X100009310A3BD7FC.avi'", "'0X100009310A3BD7FC.avi'", "'0X100009310A3BD7FC.avi'", "'0X100009310A3BD7FC.avi'"]

Does exact match exist?
0


In [23]:
def contour_area_for_frame(video_name, frame_idx, tracings):
    frame_points = tracings[
        (tracings["FileName"] == video_name) &
        (tracings["Frame"] == frame_idx)
    ]

    points = []

    for _, r in frame_points.iterrows():
        points.append([r["X1"], r["Y1"]])
        points.append([r["X2"], r["Y2"]])

    points = np.array(points, dtype=np.float32)

    # remove duplicate points
    points = np.unique(points, axis=0)

    # contour area
    area = cv2.contourArea(points)

    return area

In [24]:
def get_ed_es_frames(video_name, tracings):
    frames = get_annotated_frames(video_name, tracings)

    areas = {}

    for f in frames:
        areas[f] = contour_area_for_frame(video_name, f, tracings)

    ed_frame = max(areas, key=areas.get)  # bigger ventricle
    es_frame = min(areas, key=areas.get)  # smaller ventricle

    return ed_frame, es_frame, areas

In [25]:
row = filelist.iloc[0]

base_name = row["FileName"]
video_name = base_name + ".avi"
video_path = ECHONET_DIR / "Videos" / video_name

ed_frame, es_frame, areas = get_ed_es_frames(video_name, tracings)

print("Areas:", areas)
print("ED frame:", ed_frame)
print("ES frame:", es_frame)

Areas: {np.int64(46): 36.84584949489545, np.int64(61): 389.11028974529836}
ED frame: 61
ES frame: 46


In [26]:
ed_es_tensor = read_two_frames(video_path, [ed_frame, es_frame])

print(ed_es_tensor.shape)

torch.Size([2, 3, 224, 224])
